In [1]:
from IPython.display import HTML
HTML('''<style>.jp-Cell-inputWrapper, .input { margin-top: 0.5em; }</style>''')

# Notebook 01 — Master DataFrame (Stacked Multi-Disease)
## ENGG2112 Project MODR

This notebook merges flu + COVID + RSV per-county data into a **stacked dataset** where each row is a `(county, disease)` observation. The same demographic features apply across diseases, but each county-disease pair gets its own outcome and outbreak label.

### Stacked dataset structure
| Disease | Geography | Season | Rows |
|---|---|---|---|
| **Flu** | NY 62 + PA 67 + CT 9 + DE 3 (planning regions for CT) | 2024-25 / 2025-26 | 141 |
| **COVID** | NY 62 + PA 67 + CT 8 (old counties) + DE 3 | Jun 2022 – May 2023 (endemic) | 140 |
| **RSV** | PA 67 + CT 9 (no public NY/DE data) | 2024-25 / 2025-26 | 76 |
| **Total** | — | — | **357** |

### Why stacked instead of merged-outcome
- Different diseases have different surveillance regimes — combining cases into one number forces incompatible comparisons
- Stacking lets the model learn demographic-outbreak relationships *across* diseases
- `disease` becomes a categorical feature — model can capture disease-specific demographic effects
- Sample size grows ~2.5× from 141 to 357 (balanced disease distribution)

### Why balanced disease counts (not all 3,000+ COVID counties)
Per the methodological decision in PROJECT_JOURNAL.md, we sample COVID to ~140 rows (matched to flu) so the model isn't dominated by COVID. Goal: a "respiratory illness vulnerability" model, not a "COVID prediction" model.

### Outbreak labelling
For each row: `outbreak = 1` if county is in **top 25% by per-capita rate within (disease, state)** group. This handles different surveillance regimes per state/disease cleanly.

### Cross-validation strategy (for downstream notebooks)
Use `StratifiedGroupKFold(groups=fips)` so the same county's flu + COVID + RSV rows always go into the same fold (avoids data leakage).

### Outputs
- `data/processed/master_stacked.csv` — primary modelling dataset (~357 rows)
- `data/processed/master_counties.csv` — preserved as flu-only cross-section for backwards compatibility
- `docs/data_dictionary.md` — auto-generated column reference

## 1. Setup

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'usa_data'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DOCS_DIR = PROJECT_ROOT / 'docs'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

DTYPE = {'fips': str, 'state_fips': str}

print('Loading per-source CSVs...')

Loading per-source CSVs...


## 2. Load Disease Data Sources

In [3]:
# Flu
ny = pd.read_csv(RAW_DIR / 'ny_flu_panel.csv', dtype=DTYPE)
pa = pd.read_csv(RAW_DIR / 'pennsylvania' / 'pa_flu_panel.csv', dtype=DTYPE)
ct = pd.read_csv(RAW_DIR / 'connecticut' / 'ct_flu_panel.csv', dtype=DTYPE)
de = pd.read_csv(RAW_DIR / 'delaware' / 'de_flu_panel.csv', dtype=DTYPE)

# COVID (already pre-aggregated to county-level summary in Notebook 00)
covid = pd.read_csv(RAW_DIR / 'covid' / 'covid_endemic_2022-23.csv', dtype=DTYPE)

# RSV (PA + CT)
rsv = pd.read_csv(RAW_DIR / 'rsv' / 'rsv_panel.csv', dtype=DTYPE)

# Demographics + land
acs = pd.read_csv(RAW_DIR / 'acs_demographics' / 'acs_multistate_2022.csv', dtype=DTYPE)
land = pd.read_csv(RAW_DIR / 'acs_demographics' / 'land_area_2022.csv', dtype=DTYPE)
places = pd.read_csv(RAW_DIR / 'cdc_places' / 'places_2024_tidy.csv', dtype={'fips': str})

print(f'NY flu panel:  {ny.shape}')
print(f'PA flu panel:  {pa.shape}')
print(f'CT flu panel:  {ct.shape}')
print(f'DE flu panel:  {de.shape}')
print(f'COVID panel:   {covid.shape}')
print(f'RSV panel:     {rsv.shape}')
print(f'ACS:           {acs.shape}')
print(f'Land area:     {land.shape}')
print(f'PLACES:        {places.shape}')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/clenath24/Desktop/ENGG2112/data/raw/usa_data/ny_flu_panel.csv'

## 3. Build Flu Cross-Section

Pick most recent season per state. Same as previous Notebook 01 logic.

In [ ]:
SEASON_PER_STATE = {
    'NY': '2024-2025',
    'PA': '2025-2026',
    'CT': '2024-2025',
    'DE': '2024-2025',
}

flu_xs = pd.concat([
    ny[ny['season'] == SEASON_PER_STATE['NY']],
    pa[pa['season'] == SEASON_PER_STATE['PA']],
    ct[ct['season'] == SEASON_PER_STATE['CT']],
    de[de['season'] == SEASON_PER_STATE['DE']],
], ignore_index=True)
flu_xs['disease'] = 'FLU'
print(f'Flu cross-section: {len(flu_xs)} rows')

Flu cross-section: 141 rows


## 4. Standardise COVID Panel to Match Flu Schema

In [ ]:
# COVID needs total_cases (raw count) — derive from per-100k × pop / 100k
# But we have the per-100k metric directly which is more comparable across counties
# We'll keep total_cases_per_100k as the OUTCOME and synthesize a "total_cases" for completeness

# Map COVID columns to common schema
covid_std = pd.DataFrame({
    'state': covid['state'],
    'state_fips': covid['state_fips'],
    'fips': covid['fips'],
    'county': covid['county'],
    'season': covid['season'],
    'season_start_year': covid['season_start_year'],
    'total_cases_per_100k': covid['total_cases_per_100k'],
    'peak_week_cases_per_100k': covid['peak_week_cases_per_100k'],
    'weeks_active': covid['weeks_active'],
    'pct_weeks_high_level': covid['pct_weeks_high_level'],
    'mean_admissions_per_100k': covid['mean_admissions_per_100k'],
    'disease': 'COVID',
})
print(f'COVID standardised: {covid_std.shape}')
print(covid_std.head(2).to_string(index=False))

COVID standardised: (140, 12)
state state_fips  fips           county    season  season_start_year  total_cases_per_100k  peak_week_cases_per_100k  weeks_active  pct_weeks_high_level  mean_admissions_per_100k disease
   CT         09 09001 Fairfield County 2022-2023               2022               4313.88                    179.79          50.0                   4.0                    10.402   COVID
   CT         09 09003  Hartford County 2022-2023               2022               4729.48                    198.38          50.0                   6.0                    10.502   COVID


## 5. Standardise RSV Panel to Match Flu Schema

In [ ]:
rsv_std = rsv[['state', 'state_fips', 'fips', 'county', 'season', 'season_start_year', 'total_cases', 'disease']].copy()
print(f'RSV standardised: {rsv_std.shape}')

RSV standardised: (76, 8)


## 6. Build Stacked Multi-Disease DataFrame

Each row = (fips, disease). All diseases stacked. Population numerator differs per disease (flu=cases, COVID=cases per 100K cumulative, RSV=cases) so we'll normalise by population in the next step.

In [ ]:
# Standardise flu to have the same outcome columns as COVID/RSV
# We need flu cases as raw count (already have total_cases) AND per-100k rate (compute later)
flu_xs_std = flu_xs[['state', 'state_fips', 'fips', 'county', 'season', 'season_start_year',
                     'total_cases', 'weeks_active', 'peak_week_cases', 'time_to_peak',
                     'outbreak_steepness', 'pct_type_a', 'disease']].copy()

# RSV — same schema as flu (raw counts)
rsv_std_full = rsv_std.copy()
for col in ['weeks_active', 'peak_week_cases', 'time_to_peak', 'outbreak_steepness', 'pct_type_a',
            'total_cases_per_100k', 'peak_week_cases_per_100k', 'pct_weeks_high_level',
            'mean_admissions_per_100k']:
    if col not in rsv_std_full.columns:
        rsv_std_full[col] = np.nan

# COVID — has per-100k metrics; compute back-derived total_cases
# We'll merge with population first to do this properly. For now, keep per-100k.
covid_std_full = covid_std.copy()
for col in ['total_cases', 'weeks_active', 'peak_week_cases', 'time_to_peak', 'outbreak_steepness',
            'pct_type_a']:
    if col not in covid_std_full.columns:
        covid_std_full[col] = np.nan

# Stack
stacked = pd.concat([flu_xs_std, covid_std_full, rsv_std_full], ignore_index=True)
print(f'Stacked dataset: {len(stacked)} rows')
print(f'Per disease: {stacked["disease"].value_counts().to_dict()}')
print(f'Per (disease, state): \n{stacked.groupby(["disease", "state"]).size()}')

Stacked dataset: 357 rows
Per disease: {'FLU': 141, 'COVID': 140, 'RSV': 76}
Per (disease, state): 
disease  state
COVID    CT        8
         DE        3
         NY       62
         PA       67
FLU      CT        9
         DE        3
         NY       62
         PA       67
RSV      CT        9
         PA       67
dtype: int64


## 7. Merge Demographics & Engineer Features

In [ ]:
# Drop 'state' from acs to avoid collision (we have state in stacked already)
acs_for_merge = acs.drop(columns=['state'])

before = stacked.shape
stacked = stacked.merge(acs_for_merge, on='fips', how='left')
stacked = stacked.merge(land[['fips', 'land_area_sqmi']], on='fips', how='left')
print(f'After ACS+land merge: {before} → {stacked.shape}')

# Sanity check
n_missing = stacked['pop_total'].isna().sum()
if n_missing > 0:
    missing_fips = stacked[stacked['pop_total'].isna()][['disease', 'state', 'fips', 'county']].drop_duplicates()
    print(f'⚠️  {n_missing} rows missing demographics:')
    print(missing_fips.to_string(index=False))
else:
    print('✅ All rows have demographics')

After ACS+land merge: (357, 17) → (357, 32)
✅ All rows have demographics


In [ ]:
# Merge PLACES (will be NaN for old CT COVID rows since PLACES uses new planning regions)
stacked = stacked.merge(places.drop(columns=['state']), on='fips', how='left')
print(f'After PLACES merge: {stacked.shape}')

places_cols = ['pct_uninsured_brfss', 'pct_checkup', 'pct_obesity', 'pct_diabetes',
               'pct_copd', 'pct_chd', 'pct_csmoking']
n_missing_places = stacked[places_cols].isna().any(axis=1).sum()
print(f'\nRows missing PLACES features: {n_missing_places} (old CT COVID rows)')

# Impute missing PLACES with state-mean (so old CT COVID rows get CT mean from new planning regions)
for col in places_cols:
    state_mean = stacked.groupby('state')[col].transform('mean')
    stacked[col] = stacked[col].fillna(state_mean)

# Verify imputation worked
remaining = stacked[places_cols].isna().sum().sum()
print(f'Missing PLACES after state-mean imputation: {remaining}')

After PLACES merge: (357, 41)

Rows missing PLACES features: 8 (old CT COVID rows)
Missing PLACES after state-mean imputation: 0


In [ ]:
# Engineer rate-based features
stacked['pop_density_per_sqmi'] = (stacked['pop_total'] / stacked['land_area_sqmi']).round(2)
stacked['pct_elderly'] = (stacked['pop_elderly'] / stacked['pop_total'] * 100).round(2)
stacked['poverty_rate'] = (stacked['pop_below_poverty'] / stacked['pop_total'] * 100).round(2)
stacked['unemployment_rate'] = (stacked['unemployed'] / stacked['labour_force'] * 100).round(2)
stacked['public_transport_pct'] = (stacked['public_transport_commuters'] / stacked['total_commuters'] * 100).round(2)
stacked['pct_bachelors_plus'] = (stacked['pop_bachelors_plus'] / stacked['pop_total'] * 100).round(2)
stacked['pct_non_white'] = ((stacked['pop_total'] - stacked['pop_white']) / stacked['pop_total'] * 100).round(2)
stacked['pct_foreign_born'] = (stacked['pop_foreign_born'] / stacked['pop_total'] * 100).round(2)

# Compute outcome per disease
# Flu/RSV: have total_cases — compute per-100k
# COVID: already have total_cases_per_100k

flu_rsv_mask = stacked['disease'].isin(['FLU', 'RSV'])
stacked.loc[flu_rsv_mask, 'outcome_per_100k'] = (
    stacked.loc[flu_rsv_mask, 'total_cases'] / stacked.loc[flu_rsv_mask, 'pop_total'] * 100_000
).round(1)

covid_mask = stacked['disease'] == 'COVID'
stacked.loc[covid_mask, 'outcome_per_100k'] = stacked.loc[covid_mask, 'total_cases_per_100k'].round(1)

print('Outcome (per 100K) summary by disease:')
print(stacked.groupby('disease')['outcome_per_100k'].describe().round(0))

Outcome (per 100K) summary by disease:
         count    mean     std     min     25%     50%     75%      max
disease                                                                
COVID    140.0  5135.0  1370.0  3023.0  4234.0  4858.0  5789.0  11057.0
FLU      141.0  1694.0   861.0   216.0   966.0  1557.0  2194.0   4332.0
RSV       76.0   267.0   160.0     0.0   157.0   240.0   331.0    876.0


## 8. Outbreak Label — Top 25% Within (Disease, State)

In [ ]:
def within_disease_state_label(df, percentile=75):
    out = df.copy()
    out['outbreak'] = 0
    for (dis, state), grp in out.groupby(['disease', 'state']):
        if len(grp) < 2:
            continue
        threshold = grp['outcome_per_100k'].quantile(percentile / 100)
        idx = grp.index[grp['outcome_per_100k'] >= threshold]
        out.loc[idx, 'outbreak'] = 1
    return out

stacked = within_disease_state_label(stacked, percentile=75)

print('Outbreak distribution per (disease, state):')
print(stacked.groupby(['disease', 'state'])['outbreak'].agg(['count', 'sum', 'mean']).round(2))
print(f'\nOverall outbreak rate: {stacked["outbreak"].mean():.1%}')

Outbreak distribution per (disease, state):
               count  sum  mean
disease state                  
COVID   CT         8    2  0.25
        DE         3    1  0.33
        NY        62   16  0.26
        PA        67   17  0.25
FLU     CT         9    3  0.33
        DE         3    1  0.33
        NY        62   16  0.26
        PA        67   17  0.25
RSV     CT         9    3  0.33
        PA        67   17  0.25

Overall outbreak rate: 26.1%


## 9. Quality Checks

In [ ]:
print('=== Quality Checks ===\n')
assert len(stacked) > 0
assert stacked.duplicated(subset=['fips', 'disease']).sum() == 0, 'Duplicate (fips, disease)'
print(f'✅ {len(stacked)} unique (fips, disease) rows')

# Feature completeness
FEATURE_COLS = [
    'pop_density_per_sqmi', 'median_age', 'pct_elderly', 'avg_household_size',
    'median_income', 'poverty_rate', 'unemployment_rate', 'public_transport_pct',
    'pct_bachelors_plus', 'pct_non_white', 'pct_foreign_born',
    'pct_checkup', 'pct_obesity', 'pct_diabetes', 'pct_copd', 'pct_chd', 'pct_csmoking',
]
for col in FEATURE_COLS + ['outcome_per_100k', 'outbreak']:
    n_missing = stacked[col].isna().sum()
    if n_missing > 0:
        print(f'  ⚠️  {col}: {n_missing} missing')
    else:
        print(f'  ✅ {col}: complete')
print(f'\n{len(FEATURE_COLS)} features × {len(stacked)} obs = {len(stacked)/len(FEATURE_COLS):.1f}:1 ratio')

=== Quality Checks ===

✅ 357 unique (fips, disease) rows
  ✅ pop_density_per_sqmi: complete
  ✅ median_age: complete
  ✅ pct_elderly: complete
  ✅ avg_household_size: complete
  ✅ median_income: complete
  ✅ poverty_rate: complete
  ✅ unemployment_rate: complete
  ✅ public_transport_pct: complete
  ✅ pct_bachelors_plus: complete
  ✅ pct_non_white: complete
  ✅ pct_foreign_born: complete
  ✅ pct_checkup: complete
  ✅ pct_obesity: complete
  ✅ pct_diabetes: complete
  ✅ pct_copd: complete
  ✅ pct_chd: complete
  ✅ pct_csmoking: complete
  ✅ outcome_per_100k: complete
  ✅ outbreak: complete

17 features × 357 obs = 21.0:1 ratio


## 10. Export Stacked Master + Maintain Flu-Only Cross-Section

In [ ]:
# Stacked master — primary output for downstream notebooks
EXPORT_COLS = (
    ['state', 'state_fips', 'fips', 'county', 'NAME', 'disease', 'season', 'season_start_year']
    + ['outcome_per_100k', 'total_cases', 'total_cases_per_100k',
       'peak_week_cases', 'peak_week_cases_per_100k', 'weeks_active',
       'time_to_peak', 'outbreak_steepness', 'pct_type_a',
       'pct_weeks_high_level', 'mean_admissions_per_100k']
    + FEATURE_COLS
    + ['pop_total', 'land_area_sqmi', 'outbreak']
)
EXPORT_COLS = [c for c in EXPORT_COLS if c in stacked.columns]

stacked_export = stacked[EXPORT_COLS].copy()
stacked_export = stacked_export.sort_values(['disease', 'state', 'fips']).reset_index(drop=True)
stacked_export.to_csv(PROCESSED_DIR / 'master_stacked.csv', index=False)
print(f'✅ master_stacked.csv: {stacked_export.shape}, {(PROCESSED_DIR / "master_stacked.csv").stat().st_size / 1024:.1f} KB')

# Backwards-compat: flu-only cross-section (same shape as before)
flu_only = stacked_export[stacked_export['disease'] == 'FLU'].copy()
flu_only.to_csv(PROCESSED_DIR / 'master_counties.csv', index=False)
print(f'✅ master_counties.csv (flu-only, backward compat): {flu_only.shape}')

print(f'\nFinal stacked dataset summary:')
print(stacked_export.groupby(['disease', 'state']).agg(
    n=('fips', 'count'),
    outbreak_n=('outbreak', 'sum'),
    season=('season', 'first')
).to_string())

✅ master_stacked.csv: (357, 39), 76.9 KB
✅ master_counties.csv (flu-only, backward compat): (141, 39)

Final stacked dataset summary:
                n  outbreak_n     season
disease state                           
COVID   CT      8           2  2022-2023
        DE      3           1  2022-2023
        NY     62          16  2022-2023
        PA     67          17  2022-2023
FLU     CT      9           3  2024-2025
        DE      3           1  2024-2025
        NY     62          16  2024-2025
        PA     67          17  2025-2026
RSV     CT      9           3  2024-2025
        PA     67          17  2025-2026


## 11. Update Data Dictionary

In [ ]:
DATA_DICT_NEW = {
    'state': ('Identity', 'string', 'Two-letter state code (NY/PA/CT/DE)'),
    'state_fips': ('Identity', 'string', '2-digit state FIPS'),
    'fips': ('Identity', 'string', '5-digit county/area FIPS'),
    'county': ('Identity', 'string', 'County or planning region name'),
    'NAME': ('Identity', 'string', 'Full Census-formatted area name'),
    'disease': ('Identity', 'string', 'FLU / COVID / RSV — categorical disease type'),
    'season': ('Identity', 'string', 'Season label (e.g. 2024-2025)'),
    'season_start_year': ('Identity', 'int', 'Year season started'),

    'outcome_per_100k': ('Outcome', 'float', 'Primary outcome — disease cases per 100K population'),
    'total_cases': ('Outcome', 'float', 'Total raw case count (FLU/RSV only)'),
    'total_cases_per_100k': ('Outcome', 'float', 'Cumulative weekly cases per 100K (COVID only — endemic season Jun 2022–May 2023)'),
    'peak_week_cases': ('Outcome', 'float', 'Maximum single-week count (FLU/RSV only)'),
    'peak_week_cases_per_100k': ('Outcome', 'float', 'Highest single-week rate per 100K (COVID only)'),
    'weeks_active': ('Outcome', 'float', 'Weeks with non-zero cases'),
    'pct_weeks_high_level': ('Outcome', 'float', '% of weeks at CDC High community level (COVID only)'),

    'pop_density_per_sqmi': ('Feature', 'float', 'Population / land area (urban-ness)'),
    'median_age': ('Feature', 'float', 'ACS B01002_001E'),
    'pct_elderly': ('Feature', 'float', 'Pop 65+ / total × 100'),
    'avg_household_size': ('Feature', 'float', 'ACS B25010_001E'),
    'median_income': ('Feature', 'float', 'Median household income (USD)'),
    'poverty_rate': ('Feature', 'float', 'Below poverty / total × 100'),
    'unemployment_rate': ('Feature', 'float', 'Unemployed / labour force × 100'),
    'public_transport_pct': ('Feature', 'float', 'Public transport commuters / total × 100'),
    'pct_bachelors_plus': ('Feature', 'float', "Bachelor's+ / total × 100"),
    'pct_non_white': ('Feature', 'float', '(Total - white) / total × 100'),
    'pct_foreign_born': ('Feature', 'float', 'Foreign-born / total × 100'),

    'pct_checkup': ('Feature (PLACES)', 'float', '% adults with annual checkup — preventive care engagement'),
    'pct_obesity': ('Feature (PLACES)', 'float', '% adults obese — flu severity factor'),
    'pct_diabetes': ('Feature (PLACES)', 'float', '% adults with diabetes — comorbidity'),
    'pct_copd': ('Feature (PLACES)', 'float', '% adults with COPD — direct lung comorbidity'),
    'pct_chd': ('Feature (PLACES)', 'float', '% adults with heart disease — comorbidity'),
    'pct_csmoking': ('Feature (PLACES)', 'float', '% adults currently smoking'),

    'pop_total': ('Raw', 'int', 'Total population'),
    'land_area_sqmi': ('Raw', 'float', 'Land area (square miles)'),
    'outbreak': ('Label', 'int', 'Top 25% within (disease, state) by outcome_per_100k'),
}

dd_path = DOCS_DIR / 'data_dictionary.md'
with open(dd_path, 'w') as f:
    f.write('# Data Dictionary — `master_stacked.csv`\n\n')
    f.write('Auto-generated by Notebook 01. Each row = `(county, disease)` observation.\n\n')
    f.write(f'Shape: {stacked_export.shape[0]} rows × {stacked_export.shape[1]} cols\n')
    f.write(f'Diseases: {sorted(stacked_export["disease"].unique())}\n')
    f.write(f'States: {sorted(stacked_export["state"].unique())}\n\n')
    f.write('See `REFERENCES.md` for source citations.\n\n')
    
    sections = ['Identity', 'Outcome', 'Feature', 'Feature (PLACES)', 'Raw', 'Label']
    for sect in sections:
        cols_in_sect = [(c, info) for c, info in DATA_DICT_NEW.items() if info[0] == sect and c in stacked_export.columns]
        if not cols_in_sect:
            continue
        f.write(f'## {sect}\n\n| Column | Type | Description |\n|---|---|---|\n')
        for col, (_, dtype, desc) in cols_in_sect:
            f.write(f'| `{col}` | {dtype} | {desc} |\n')
        f.write('\n')

print(f'✅ Data dictionary: {dd_path.relative_to(PROJECT_ROOT)} ({dd_path.stat().st_size / 1024:.1f} KB)')

✅ Data dictionary: docs/data_dictionary.md (2.8 KB)


## 12. Summary

In [ ]:
print('=== Master DataFrame Construction Complete ===\n')
print(f'📊 Stacked dataset (master_stacked.csv):')
print(f'   Total rows: {len(stacked_export)}')
print(f'   Diseases:   {dict(stacked_export["disease"].value_counts())}')
print(f'   States:     {dict(stacked_export["state"].value_counts())}')
print(f'   Outbreak rate: {stacked_export["outbreak"].mean():.1%}')

print(f'\n📁 Files written:')
for f in [PROCESSED_DIR / 'master_stacked.csv', PROCESSED_DIR / 'master_counties.csv', DOCS_DIR / 'data_dictionary.md']:
    if f.exists():
        print(f'   ✅ {f.relative_to(PROJECT_ROOT)} ({f.stat().st_size / 1024:.1f} KB)')

print(f'\n👉 Next: Notebook 02 (EDA), Notebook 03 (Feature Selection), Notebook 04 (Logistic).')
print(f'   Models will use master_stacked.csv. Use StratifiedGroupKFold(groups=fips) for CV.')

=== Master DataFrame Construction Complete ===

📊 Stacked dataset (master_stacked.csv):
   Total rows: 357
   Diseases:   {'FLU': 141, 'COVID': 140, 'RSV': 76}
   States:     {'PA': 201, 'NY': 124, 'CT': 26, 'DE': 6}
   Outbreak rate: 26.1%

📁 Files written:
   ✅ data/processed/master_stacked.csv (76.9 KB)
   ✅ data/processed/master_counties.csv (29.4 KB)
   ✅ docs/data_dictionary.md (2.8 KB)

👉 Next: Notebook 02 (EDA), Notebook 03 (Feature Selection), Notebook 04 (Logistic).
   Models will use master_stacked.csv. Use StratifiedGroupKFold(groups=fips) for CV.
